# Estudio con distribución binomial

En este notebook se ejecuta y comenta la ejecución del experimento con brazos cuya recompensa viene dada por una función de distribución binomial.

## Constantes

In [1]:
SEED = 42

K = 10
RUNS = 1000
STEPS = 20_000

SAVE_PLOTS_PATH = "plots/binomial_dist"

## Librerías

In [2]:
import sys
import random

import numpy as np
from typing import List

sys.path.append('./src')
print(sys.path)

from arms import ArmBinomial, Bandit
from experiment import run_experiment
from algorithms import Algorithm, EpsilonGreedy, SoftMax, UCB1
from plotting import plot_average_rewards, plot_optimal_selections, plot_regret, plot_arm_statistics

['/home/denis/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python312.zip', '/home/denis/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12', '/home/denis/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/lib-dynload', '', '/home/denis/Documents/mia-um/eml/dourado-minguell-eml-mia-um-p1/.venv/lib/python3.12/site-packages', './src']


## Reproducibilidad

In [3]:
random.seed(SEED)
np.random.seed(SEED)

Asegurar que la generación de números aleatorios es igual para todos los experimentos.

## Experimentación

Para cada algiritmo $\epsilon$-greedy, softmax y UCB1, se ejecuta la experimentación del problema del bandido de k-brazos durante el número de pasos y ejecuciones definido en las constantes. Posteriormente, para cada algoritmo se realiza el pertienente estudio de resultados mediantes las gráficas de recompensas promedio, selecciones óptimas, arrepentimiento y estadísticas por brazo.

In [4]:
# Creación del bandido de k-brazos
bandit = Bandit(arms=ArmBinomial.generate_arms(K))
optimal_arm = bandit.optimal_arm

print(bandit)
print(f"Optimal arm: {optimal_arm + 1} with expected reward={bandit.get_expected_value(optimal_arm)}")

Bandit with 10 arms: ArmBinomial (n=100, p=0.39), ArmBinomial (n=100, p=0.91), ArmBinomial (n=100, p=0.71), ArmBinomial (n=100, p=0.19), ArmBinomial (n=100, p=0.59), ArmBinomial (n=100, p=0.1), ArmBinomial (n=100, p=0.83), ArmBinomial (n=100, p=0.69), ArmBinomial (n=100, p=0.07), ArmBinomial (n=100, p=0.92)
Optimal arm: 10 with expected reward=92.0


Para le experimentación, se va emplear un bandido de 10 brazos. Para este caso reproducible, el brazo óptimo es el número 8 con una recomensa media esperada de 9.56.

### Algoritmo $\epsilon$-greedy

En el algoritmo $\epsilon$-greedy se van a comparar los resultados con distintos valores para epsilon:
 - $\epsilon = 0.0$
 - $\epsilon = 0.01$
 - $\epsilon = 0.1$
 - $\epsilon = 0.5$

In [5]:
##### Definir los algoritmos greedy con diferente epsilon
algorithms_greedy = [
    EpsilonGreedy(k=K, epsilon=0),
    EpsilonGreedy(k=K, epsilon=0.01),
    EpsilonGreedy(k=K, epsilon=0.1),
    EpsilonGreedy(k=K, epsilon=0.5)
]

# Ejecutar el experimento
rewards_greedy, optimal_selections_greedy, regret_accumulated_greedy, arm_stats_greedy = run_experiment(bandit, algorithms_greedy, STEPS, RUNS)

KeyboardInterrupt: 

Ya se han definido los algoritmo y ejecutada la experimentación. A continuación, se van a comentar los resultados obetnidos mediante las correspondientes gráficas,

#### Recompensas promedio

In [ ]:
plot_average_rewards(STEPS, rewards_greedy, algorithms_greedy, SAVE_PLOTS_PATH)

En la gráfica de recompensa promedio vemos que hay varías configuraciones que alcanzan una recompensa en torno al valor del brazo óptimo (9.56): 0, 0.01 y 0.1, siendo algo menor la del algoritmo con epsilon=0.1 debido a un 10% de elecciones aleatorias. Esta disminución de la recompensa promedio debido a la aleatoriedad es todavía más acusada para epsilon=0.5. El número de pasos para alcanzar este valor de recompensa varía siendo dependiente de la componente exploratoria.

Destaca el valor alcanzado con epsilon=0, donde el algoritmo ha sido capaz de encontrar en la primera tirada inicial de todos los brazos uno con tanta recompensa promedio como la alcanza en más pasos por otros epsilon. En este caso se ha obtenido un buen resultado con esa estrategia, pero en caso de haber selección un brazo con peores recompensas, nunca podría haber cambiado la elección a lo largo de los pasos.

#### Selecciones óptimas

In [ ]:
plot_optimal_selections(STEPS, optimal_selections_greedy, algorithms_greedy, SAVE_PLOTS_PATH)

En la gráfica de selecciones óptimas vemos nuevamente como el algoritmo con epsilon=0 es el que más rápido crece alcanzando su estado estacionario en muy pocas iteraciones. No obstante se estanca en un valor entre 0.8 y 0.9, de esto extraemos que no ha elegido el brazo óptimo en la inicialización. De hecho, todos los algoritmos que presenta una componente exploratoria menos epsilon=0.5 acaban teniendo una mejor tasa de éxito. En el caso de epsilon=0.1, vemos que se estabiliza sobre 0.9, lo cual encaja con un aprendizaje del mejor brazo donde aproximadamente el 10% de error corresponde a los intentos aleatorios por la componente exploratoria. El algoritmo con epsilon=0.01 sigue mostrando una ligera pendiente al acabar la ejecución, siendo esperable que acabase estabilizándose en torno a 0.99. Para epsilon=0.5, la mitad de las selecciones se realizan de manera aleatoria por lo que el resultado obtenido es el esperado con un 55% de selecciones correctas: 50% de selecciones intentando maximizar la recompensa + el 5% correspondiente a las elecciones aleatorias (el 10% de 50%).

El aumento drástico del porcentaje de selecciones óptimas para epsilon=0 justo al inicio corresponde con la selección del brazo óptimo en el barrido inicial, que siempre se encuentra en la misma posición, la octava en este caso.

#### Arrepentimiento

In [ ]:
plot_regret(STEPS, regret_accumulated_greedy, algorithms_greedy, SAVE_PLOTS_PATH)

En la gráfica de arrepentimiento acumulado, apreciamos como este aumenta a distintas velocidades dependiendo del valor epsilon. El algoritmo con epsilon=0 comienza y termina con la menor tasa de arrepentimiento acumulado por la decisión de un brazo con un arrepentimiento bajo. Siguiendo con lo visto en las gráficas anteriores, el arrepentimiento para epsilon=0.5 asciende de forma lineal mientras que para epsilon=0.01 y epsilon=0.1 el aumento es más controlado debido a su menor exploración. Al final de los pasos, vemos como el arrepentimiento acumulado de epsilon=0.1 supera al de eplison=0.01 con una tendencia alcista mayor que este último.

#### Estadísticas por brazo

In [ ]:
plot_arm_statistics(arm_stats_greedy, algorithms_greedy, SAVE_PLOTS_PATH)

Con estos diagramas de estadísticas por brazo, se puede confirmar que el algoritmo con epsilon=0 en la gran mayoría de sus ejecuciones ha elegido el más óptimo o el segundo más óptimo, de ahí los buenos resultados obtenidos pese a la nula exploración. El resto de algoritmos también toman el brazo óptimo el que más. Cuanto menor es la exploración, más veces han elegido el mejor brazo debido a la larga cantidad de pasos donde explotar el brazo con mejor promedio.

### Algoritmo UCB1

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 0.1. Se realizan 500 ejecuciones de ~1000~5000 pasos cada una.
Se contrastan:
 - UCB1 con valor $exploration = 0$
 - UCB1 con valor $exploration = 0.1$
 - UCB1 con valor $exploration = 1$
 - UCB1 con valor $exploration = 5$
 - UCB1 con valor $exploration = 10$

In [ ]:
# Definir los algoritmos softmax con diferente temperatura
algorithms_ucb1 = [
    UCB1(k=K, exploration=0),
    UCB1(k=K, exploration=0.1),
    UCB1(k=K, exploration=1),
    UCB1(k=K, exploration=5)
]

# Ejecutar el experimento
rewards_ucb1, optimal_selections_ucb1, regret_accumulated_ucb1, arm_stats_ucb1 = run_experiment(bandit, algorithms_ucb1, STEPS, RUNS)

Desde un primer vistazo el comportamiento de todos los algoritmos USB1 sorprende por su recompensa promedio. Vemos como los distintos parametros empleados para la variante exploration parecen afectar en menor medida que para los otros algoritmos del estudio. En este caso todas las versiones alcanzan un valor de recompensa promedio muy proximo al valor optimo, tan solo afectando a la velocidad con la que se alcanza esa precisión.


En las dos siguientes gráficas se aprecian más diferencias que permiten valorar de manera más apropiada su comportamiento. En el caso de selecciones optimas vemos como cuando el termino  exploration = 0 el algoritmo no presenta componente exploratoria y, por tanto, se comporta similar que los otros dos algoritmos: alcanzando rapidamente una precisión del ~85% pero no aprendiendo con el paso de las iteraciones. Lo mismo ocurre con exploration = 0.1 y 1 pero con un mayor porcentaje de acierto en cada caso, alcanzando el ~100% de acierto en el segundo caso. Por su parte, las versiones con mayor componente exploratoria tardan más en aprender.

Este menor ritmo de aprendizaje se refleja en un mayor arrepentimiento acumulado frente a las otras alternativas, siendo nuevamente exploration = 1 la variante que presenta un mejor comportamiento.

#### Recompensas promedio

In [ ]:
plot_average_rewards(STEPS, rewards_ucb1, algorithms_ucb1, SAVE_PLOTS_PATH)

#### Selecciones óptimas

In [ ]:
plot_optimal_selections(STEPS, optimal_selections_ucb1, algorithms_ucb1, SAVE_PLOTS_PATH)

Tras esta primera sección vamos a realizar la comprobación con los algoritmos que mejor comportamiento presentan a 5000 iteraciones. Para realizar este elección se va a realizar una elección basada en la recompensa promedio y el arrepentimiento acumulado:

 - epsilon greedy con valor $\epsilon = 0$
 - epsilon greedy con valor $\epsilon = 0.1$
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 0.1$
 - UCB1 con valor $exploration = 0.1$
 - UCB1 con valor $exploration = 1$

#### Arrepentimiento

In [ ]:
plot_regret(STEPS, regret_accumulated_ucb1, algorithms_ucb1, SAVE_PLOTS_PATH)

#### Estadísticas por brazo

In [ ]:
plot_arm_statistics(arm_stats_ucb1, algorithms_ucb1, SAVE_PLOTS_PATH)

### Algoritmo SoftMax

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 0.1. Se realizan 500 ejecuciones de  ~1000~5000 pasos cada una.
Se contrastan:
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 0.1$
 - SoftMax con valor $temperature = 1$
 - SoftMax con valor $temperature = 10$
 - SoftMax con valor $temperature = 100$

In [ ]:
# Definir los algoritmos softmax con diferente temperatura
algorithms_softmax = [
    SoftMax(k=K, temperature=0.001),
    SoftMax(k=K, temperature=0.1),
    SoftMax(k=K, temperature=1),
    SoftMax(k=K, temperature=5)
]

# Ejecutar el experimento
rewards_softmax, optimal_selections_softmax, regret_accumulated_softmax, arm_stats_softmax = run_experiment(bandit, algorithms_softmax, STEPS, RUNS)

#### Recompensas promedio

In [ ]:
plot_average_rewards(STEPS, rewards_softmax, algorithms_softmax, SAVE_PLOTS_PATH)

En el caso del algoritmo SoftMax nos encontramos con un comportamiento diferente al $\epsilon$-greedy donde, tras un primer barrido por todos los brazos, el comportamiento se mantiene estable en el tiempo, mostrando gráficas planas tanto en "recompensa promedio" como en "selecciones optimas". En esta segunda gráfica tan solo se aprecia cierta evolución para temperature = 0.1.

Entre las dos configuraciones que presentan mejor comportamiento (temperature = 0.001 y 0.1) los resultados obtenidos con temperature = 0.1 son ligeramente superiores pero no de una manera claramente destacable (a excepcion de el porcentaje de selecciones optimas". Esto puede deberse de la misma manera a lo ocurrido al algoritmo de $\epsilon$-greedy, donde una componente exploratoria nula o practicamente nula puede llevar a la elección de un brazo suboptimo (los valores obtenidos para el valor más pequeño de temperatura son muy similares a epsilon = 0).
Viendo las gráficas de recompensa promedio, vemos como los brazos 6 (mu=8.8) y 7 (mu=9.56) tienen valores lo suficientemente próximos como para que, con la desviacion estandar dada, presenten distribuciones que se superpongan y el de menor valor medio de una recompensa mayor al de valor medio más alto.

#### Selecciones óptimas

In [ ]:
plot_optimal_selections(STEPS, optimal_selections_softmax, algorithms_softmax, SAVE_PLOTS_PATH)

#### Arrepentimiento

In [ ]:
plot_regret(STEPS, regret_accumulated_softmax, algorithms_softmax, SAVE_PLOTS_PATH)

#### Estadísticas por brazo

In [ ]:
plot_arm_statistics(arm_stats_softmax, algorithms_softmax, SAVE_PLOTS_PATH)

In [ ]:
raise Exception("")

### Brazos = 10 ---- Desviacion = 0.1

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 0.1. Se realizan 500 ejecuciones de 1000 pasos cada una.<br>
Se contrastan:

 - epsilon greedy con valor $\epsilon = 0$
 - epsilon greedy con valor $\epsilon = 0.1$
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 0.1$
 - UCB1 con valor $exploration = 0.1$
 - UCB1 con valor $exploration = 1$

In [ ]:

# Parámetros del experimento
seed = 42
np.random.seed(seed)  # Fijar la semilla para reproducibilidad

k = 10  # Número de brazos
steps = 1000  # Número de pasos que se ejecutarán cada algoritmo
runs = 500  # Número de ejecuciones

# Creación del bandit
bandit_10_100 = Bandit(arms=ArmNormal.generate_arms(k, sigma = 0.1)) # Generar un bandido con k brazos de distribución normal
print(bandit_10_100)

optimal_arm_10_100 = bandit_10_100.optimal_arm
print(f"Optimal arm: {optimal_arm_10_100 + 1} with expected reward={bandit_10_100.get_expected_value(optimal_arm_10_100)}")


In [ ]:

# Definir los algoritmos a comparar. En este caso son 3 algoritmos epsilon-greedy con diferentes valores de epsilon.
algorithms_10_100 = [
    EpsilonGreedy(k=k, epsilon=0),
    EpsilonGreedy(k=k, epsilon=0.1),
    SoftMax(k=k, temperature=0.001),
    SoftMax(k=k, temperature=0.1),
    UCB1(k=k, exploration=0.1),
    UCB1(k=k, exploration=1)
    ]

# Ejecutar el experimento y obtener las recompensas promedio y promedio de las selecciones óptimas
rewards_10_100, optimal_selections_10_100, regret_accumulated_10_100, arm_stats_10_100 = run_experiment(bandit_10_100, algorithms_10_100, steps, runs)

generate_plots(steps, rewards_10_100, algorithms_10_100, optimal_selections_10_100, regret_accumulated_10_100, arm_stats_10_100)

En esta primera ejecución se emplea un conjunto de recompensas con una desviación estandar de 0.1.<br>

Los resultados obtenidos muestran como varios de los algoritmos presentan un comportamiento identico, con las lineas de las gráficas superponiendose entre ellas.

Esto es debido al valor de desviación empleado, ocasionando que las distribuciones de probabilidad que presenta cada brazo se encuentren totalment separadas. De esta manera, tras un primer barrido por todos los brazos, el algoritmo es capaz de distinguir que brazo es el que tiene una recompensa más alta con total fiabilidad.

La gráfica más diferenciada es precisamente para el algoritmo de $\epsilon$-greedy que presenta una aleatoriedad del 10%.

Analizando las gráficas de recompensa promedio por brazo vemos como todos los algoritmos han escogido los mismos brazos con desviaciones míminas en el numero de veces.

### Brazos = 10 ---- Desviacion = 1.0

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 1. Se realizan 500 ejecuciones de 1000 pasos cada una.
Se contrastan:
 - epsilon greedy con valor $\epsilon = 0.0$
 - epsilon greedy con valor $\epsilon = 0.1$
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 0.1$
 - UCB1 con valor $exploration = 0.1$
 - UCB1 con valor $exploration = 1$

In [ ]:
# Parámetros del experimento
seed = 42
np.random.seed(seed)  # Fijar la semilla para reproducibilidad

k = 10  # Número de brazos
steps = 1000  # Número de pasos que se ejecutarán cada algoritmo
runs = 500  # Número de ejecuciones

# Creación del bandit
bandit_10_1 = Bandit(arms=ArmNormal.generate_arms(k, sigma = 1.0)) # Generar un bandido con k brazos de distribución normal
print(bandit_10_1)

optimal_arm_10_1 = bandit_10_1.optimal_arm
print(f"Optimal arm: {optimal_arm_10_1 + 1} with expected reward={bandit_10_1.get_expected_value(optimal_arm_10_1)}")

# Definir los algoritmos a comparar. En este caso son 3 algoritmos epsilon-greedy con diferentes valores de epsilon.
algorithms_10_1 = [EpsilonGreedy(k=k, epsilon=0),
              EpsilonGreedy(k=k, epsilon=0.1),
              SoftMax(k=k, temperature=0.001),
              SoftMax(k=k, temperature=0.1),
              UCB1(k=k, exploration=0.1),
              UCB1(k=k, exploration=1)
              ]

# Ejecutar el experimento y obtener las recompensas promedio y promedio de las selecciones óptimas
rewards_10_1, optimal_selections_10_1, regret_accumulated_10_1, arm_stats_10_1 = run_experiment(bandit_10_1, algorithms_10_1, steps, runs)

generate_plots(steps, rewards_10_1, algorithms_10_1, optimal_selections_10_1, regret_accumulated_10_1, arm_stats_10_1)


Al aumentar la desviacion de las recompensas los resultados son los esperados tras la primera sección exploratoria, con el algoritmo UCB1 con adjust = 1 destacando sobre los demas alcanzando una precisión en la elección del brazo optimo de casi el 100%, escogiendo de media el brazo optimo en 966 de las 1000 iteraciones.

### Brazos = 10 ---- Desviacion = 5.0

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 5. Se realizan 500 ejecuciones de 1000 pasos cada una.
Se contrastan:
 - epsilon greedy con valor $\epsilon = 0.0$
 - epsilon greedy con valor $\epsilon = 0.1$
 - UCB1 con valor $exploration = 1$
 - UCB1 con valor $exploration = 5$
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 10000$

In [ ]:
# Parámetros del experimento
seed = 42
np.random.seed(seed)  # Fijar la semilla para reproducibilidad

k = 10  # Número de brazos
steps = 1000  # Número de pasos que se ejecutarán cada algoritmo
runs = 500  # Número de ejecuciones

# Creación del bandit
bandit_10_5 = Bandit(arms=ArmNormal.generate_arms(k, sigma = 5.0)) # Generar un bandido con k brazos de distribución normal
print(bandit_10_5)

optimal_arm_10_5 = bandit_10_5.optimal_arm
print(f"Optimal arm: {optimal_arm_10_5 + 1} with expected reward={bandit_10_5.get_expected_value(optimal_arm_10_5)}")

# Definir los algoritmos a comparar. En este caso son 3 algoritmos epsilon-greedy con diferentes valores de epsilon.
algorithms_10_5 = [EpsilonGreedy(k=k, epsilon=0),
              EpsilonGreedy(k=k, epsilon=0.1),
              SoftMax(k=k, temperature=0.001),
              SoftMax(k=k, temperature=0.1),
              UCB1(k=k, exploration=0.1),
              UCB1(k=k, exploration=1)
              ]

# Ejecutar el experimento y obtener las recompensas promedio y promedio de las selecciones óptimas
rewards_10_5, optimal_selections_10_5, regret_accumulated_10_5, arm_stats_10_5 = run_experiment(bandit_10_5, algorithms_10_5, steps, runs)

generate_plots(steps, rewards_10_5, algorithms_10_5, optimal_selections_10_5, regret_accumulated_10_5, arm_stats_10_5)

Al aumentar la desviación estandar de cada uno de los brazos, vemos como las recompensas promedio de los diferentes algoritmos se presentan superpuestas en la gráfica. En cambió la gráfica de selecicones optimas sí que muestra diferentes comportamientos en los algoritmos. El motivo de esta discrepancia se debe a $\sigma$ = 5. Este valor hace que las distribuciones de probabilidad de muchos brazos se superpongan entre ellas haciendo que, a pesar de realizar selecciones equivocadas, podamos obtener recompensas bastante elevadas.

Esto se aprecia en los graficos de columnas. En ellos se ve como todos los algoritmos seleccionan al menos 2 brazos con bastante frecuencia debido a esas distribuciones superpuestas, evitando tan solo aquellos que presentan valores muy inferiores.

### Brazos = 10 ---- Desviacion = 100

Para acabar de corroborar la influencia de $\sigma$ se realiza una nueva ejecucion con $\sigma$ = 100. En este caso, dado el rango de valores medios que pueden presentar los brazos, las distribuciones de las recompensas serán practicamente coincidentes ocasionando una seleccion muy homogénea de todos los brazos.

Se realiza el experimento usando 10 brazos, cada uno de acuerdo a una distribución gaussina con desviación 100. Se realizan 500 ejecuciones de 1000 pasos cada una.
Se contrastan:
 - epsilon greedy con valor $\epsilon = 0.0$
 - epsilon greedy con valor $\epsilon = 0.1$
 - UCB1 con valor $exploration = 1$
 - UCB1 con valor $exploration = 5$
 - SoftMax con valor $temperature = 0.001$
 - SoftMax con valor $temperature = 10000$

In [ ]:

# Parámetros del experimento
seed = 42
np.random.seed(seed)  # Fijar la semilla para reproducibilidad

k = 10  # Número de brazos
steps = 1000  # Número de pasos que se ejecutarán cada algoritmo
runs = 500  # Número de ejecuciones

# Creación del bandit
bandit_10_100 = Bandit(arms=ArmNormal.generate_arms(k, sigma = 100)) # Generar un bandido con k brazos de distribución normal
print(bandit_10_100)

optimal_arm_10_100 = bandit_10_100.optimal_arm
print(f"Optimal arm: {optimal_arm_10_100 + 1} with expected reward={bandit_10_100.get_expected_value(optimal_arm_10_100)}")

# Definir los algoritmos a comparar. En este caso son 3 algoritmos epsilon-greedy con diferentes valores de epsilon.
algorithms_10_100 = [EpsilonGreedy(k=k, epsilon=0),
              EpsilonGreedy(k=k, epsilon=0.1),
              SoftMax(k=k, temperature=0.001),
              SoftMax(k=k, temperature=0.1),
              UCB1(k=k, exploration=0.1),
              UCB1(k=k, exploration=1)
              ]

# Ejecutar el experimento y obtener las recompensas promedio y promedio de las selecciones óptimas
rewards_10_100, optimal_selections_10_100, regret_accumulated_10_100, arm_stats_10_100 = run_experiment(bandit_10_100, algorithms_10_100, steps, runs)

generate_plots(steps, rewards_10_100, algorithms_10_100, optimal_selections_10_100, regret_accumulated_10_100, arm_stats_10_100)

Tal y como predecíamos antes de la ejecución, al presentar una desviación estandar tan grande el sistema es incapaz de realizar una seleccion apropiada debido al "ruido" con el que obtiene recompensas, siendo incapaz de extraer la información necearia para realizar las selecciones de manera apropiada.

Si bien presenta cierta preferencia por el brazo optimo, solo lo selecciona aproximadamente un 13% de las ocasiones (sobre un 10% en caso de selecciones 100% aleatorias). El caso más destacado es para $\epsilon$-greedy = 0, donde el brazo óptimo es seleccionado tan solo 111 veces, con 3 brazos siendo seleccionados el mismo numero de veces o más. Otros dos brazos son seleccionados 106 y 108 veces con recompensas medias de 6.47 y 7.25 respectivamente frente al 9.9 de recompensa media del brazo óptimo.

 ### Análisis detallado de la imagen

La imagen muestra un gráfico de líneas titulado **"Recompensa Promedio vs Pasos de Tiempo"**, donde se analiza el desempeño de diferentes estrategias del algoritmo **ε-Greedy** en un entorno de multi-armed bandit. En el eje **x** se representan los **pasos de tiempo**, mientras que en el eje **y** se muestra la **recompensa promedio** obtenida por cada algoritmo.


1. **Tres líneas de colores distintos representan diferentes valores de ε en el algoritmo ε-Greedy:**
   - **Azul (ε = 0):** Representa una estrategia completamente **explotadora**, es decir, que siempre elige la acción que ha dado la mejor recompensa hasta ahora sin explorar nuevas opciones.
   - **Naranja (ε = 0.01):** Representa una estrategia con una pequeña probabilidad del 1% de elegir una acción aleatoria (exploración).
   - **Verde (ε = 0.1):** Representa una estrategia con un 10% de probabilidad de explorar acciones aleatorias.

2. **Crecimiento de la recompensa promedio:**
   - La línea **verde (ε=0.1)** alcanza rápidamente una recompensa promedio alta, lo que indica que la estrategia con mayor exploración aprende más rápido qué brazos del bandit son óptimos.
   - La línea **naranja (ε=0.01)** también muestra un crecimiento, pero más lento en comparación con ε=0.1.
   - La línea **azul (ε=0)** se mantiene en un nivel bajo de recompensa, lo que sugiere que no logra encontrar el mejor brazo porque no explora nuevas opciones.

---





## Conclusiones

Hemos estudiado un  **experimento de toma de decisiones secuenciales**, modelado con un **Multi-Armed Bandit (MAB)**. Este problema es fundamental en el aprendizaje por refuerzo y la teoría de decisiones. La idea principal es que un agente debe aprender cuál es la mejor acción (brazo del bandit) a partir de la experiencia acumulada. Para este estudio nos hemos centrado solo en el estudio del algoritmo epsilon-greedy, llegando a las siguientes conclusiones a partir de los resultados obtenidos y la gráfica generada:

#### **1. Exploración vs Explotación**
El algoritmo **ε-Greedy** equilibra la exploración y la explotación:
- **Explotación (ε=0)**: Siempre elige la mejor opción conocida, pero si inicialmente se selecciona un brazo subóptimo, nunca descubrirá otras opciones más rentables.
- **Exploración (ε>0)**: Introduce aleatoriedad en la selección de acciones para descubrir nuevas opciones potencialmente mejores.

El gráfico confirma este comportamiento:
- **ε=0.1 (verde)** obtiene la mejor recompensa promedio a lo largo del tiempo porque explora lo suficiente como para encontrar rápidamente el mejor brazo.
- **ε=0.01 (naranja)** explora menos, por lo que tarda más en converger a una recompensa alta.
- **ε=0 (azul)** no explora en absoluto y queda atrapado en una recompensa subóptima.

#### **2. Convergencia de los algoritmos**
Los algoritmos con mayor exploración (ε=0.1) alcanzan una recompensa alta más rápido. Esto se debe a que:
- Al principio, el algoritmo **no tiene información suficiente** sobre cuál es el mejor brazo.
- Con el tiempo, al realizar exploraciones, descubre cuál es el mejor brazo y empieza a explotarlo más.
- Un **balance entre exploración y explotación** es clave para maximizar la recompensa a largo plazo.


#### **3. Aplicaciones y conclusiones**
- En problemas de toma de decisiones **(ejemplo: recomendaciones, optimización de anuncios, medicina personalizada)**, una estrategia de exploración moderada como **ε=0.1** es más efectiva para encontrar la mejor opción rápidamente.
- **La falta de exploración (ε=0)** lleva a un desempeño deficiente, ya que el agente puede quedarse atrapado en una elección subóptima.

En conclusión, **el gráfico muestra cómo un nivel adecuado de exploración mejora significativamente el rendimiento del algoritmo en un entorno de aprendizaje por refuerzo**. 🚀